# 📚 Python 滑动窗口专题 — LeetCode 复习笔记

> **覆盖题目**：LC 643 · LC 438 · LC 3 · LC 1004 · LC 209 · LC 76  
> **核心框架**：固定窗口 · 变长窗口（求最长）· 变长窗口（求最短）

| 编号 | 题名 | 窗口类型 | 难度 |
|------|------|---------|------|
| 643  | 子数组最大平均数 I | 固定窗口 | 🟢 |
| 438  | 找到字符串中所有字母异位词 | 固定窗口 | 🟡 |
| 3    | 无重复字符的最长子串 | 变长窗口（求最长）| 🟡 |
| 1004 | 最大连续1的个数 III | 变长窗口（求最长）| 🟡 |
| 209  | 长度最小的子数组 | 变长窗口（求最短）| 🟡 |
| 76   | 最小覆盖子串 | 变长窗口（求最短）| 🔴 |

---

## 滑动窗口三类范式对照

| 类型 | 代表题 | 窗口大小 | 左指针何时移动 | 何时更新答案 | 初始答案 |
|------|--------|---------|--------------|------------|---------|
| **固定窗口** | LC 643, 438 | 固定 k | 每步**同步** +1 | 每次右移后 | 首窗口 |
| **变长-求最长** | LC 3, 1004 | 动态 | 条件**不满足**时收缩 | 收缩后（窗口合法时）| 0，取 `max` |
| **变长-求最短** | LC 209, 76 | 动态 | 条件**满足**时收缩 | 收缩**前**（满足时立刻记录）| ∞，取 `min` |

> **核心口诀**：  
> - 求最长 → 窗口非法才收缩，收缩后取 max  
> - 求最短 → 窗口合法就收缩，收缩前取 min

In [ ]:
from typing import List
from collections import Counter
print("环境就绪 ✅")

---
## Part 1 — 固定大小滑动窗口

**框架**：窗口大小始终等于 k，右指针扫描时左指针同步跟进。

```python
# 初始化首个窗口
window_val = init(arr[:k])

for right in range(k, n):
    window_val += 入(arr[right])      # 加入新元素
    window_val -= 出(arr[right - k])  # 移出最老元素
    更新答案
```

**触发信号**：  
- "连续 k 个元素的…最大/最小/平均…"  
- 字符串中固定长度窗口的性质（字母异位词等）

### 1. LC 643 — 子数组最大平均数 I

| 项目 | 内容 |
|------|------|
| **题型** | 固定大小滑动窗口 |
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ k ≤ n ≤ 10⁵，−10⁴ ≤ nums[i] ≤ 10⁴ |

### ⚡ 触发条件
1. "**长度恰好为 k** 的连续子数组"
2. 求子数组的**最大/最小平均值（或总和）**

### 🧠 算法
```
sum_k = sum(nums[:k])          # 初始化首个窗口
max_sum = sum_k

for i in range(k, n):
    sum_k += nums[i] - nums[i-k]   # 右进左出
    max_sum = max(max_sum, sum_k)

return max_sum / k
```
等价思路：窗口和的最大值 / k = 平均值的最大值（单调关系，不影响比较）

In [ ]:
class Solution_643:
    def findMaxAverage(self, nums: List[int], k: int) -> float:
        window_sum = sum(nums[:k])
        max_sum = window_sum
        for i in range(k, len(nums)):
            window_sum += nums[i] - nums[i - k]   # 右进左出
            max_sum = max(max_sum, window_sum)
        return max_sum / k

sol643 = Solution_643()
cases = [([1,12,-5,-6,50,3], 4, 12.75),   # 窗口[12,-5,-6,50]=51，51/4=12.75
         ([5],                1, 5.0),
         ([-1],               1, -1.0),
         ([0,1,1,3,3],        4, 2.0)]
for nums, k, exp in cases:
    r = sol643.findMaxAverage(nums, k)
    ok = abs(r - exp) < 1e-5
    print(f"{'✅' if ok else '❌'}  nums={nums}, k={k} → {r:.5f} (期望 {exp})")

### 2. LC 438 — 找到字符串中所有字母异位词

| 项目 | 内容 |
|------|------|
| **题型** | 固定大小滑动窗口 + 字符频率匹配 |
| **时间复杂度** | O(n)，Counter 比较 O(26) = O(1) |
| **空间复杂度** | O(1)（字母表大小固定为 26）|
| **数据范围** | 1 ≤ s.length, p.length ≤ 3×10⁴，仅小写字母 |

### ⚡ 触发条件
1. 字符串中查找固定长度 `len(p)` 的窗口
2. 判断窗口内字符**频率**是否与目标匹配（异位词 = 频率相同）
3. 返回**所有**满足条件的起始下标

### 🧠 算法
```
p_count = Counter(p)
window  = Counter(s[:len(p)])
result = [0] if window == p_count else []

for i in range(len(p), len(s)):
    window[s[i]] += 1                    # 右进
    window[s[i - len(p)]] -= 1          # 左出
    if window[s[i - len(p)]] == 0:
        del window[s[i - len(p)]]
    if window == p_count:
        result.append(i - len(p) + 1)   # 窗口起始下标
```
> `Counter == Counter` 每次 O(26)，总体仍为 O(n)。
> 若追求更优，可维护 `satisfied` 计数器（见注释），但代码更复杂，竞赛中通常不必要。

In [ ]:
class Solution_438:
    def findAnagrams(self, s: str, p: str) -> List[int]:
        if len(p) > len(s): return []

        p_count = Counter(p)
        window  = Counter(s[:len(p)])   # 初始化首个窗口
        result  = [0] if window == p_count else []

        for i in range(len(p), len(s)):
            # 右侧进入
            window[s[i]] += 1
            # 左侧移出
            left_char = s[i - len(p)]
            window[left_char] -= 1
            if window[left_char] == 0:
                del window[left_char]
            # 检查是否为异位词
            if window == p_count:
                result.append(i - len(p) + 1)

        return result

sol438 = Solution_438()
cases = [("cbaebabacd", "abc", [0, 6]),
         ("abab",       "ab",  [0, 1, 2]),
         ("aa",         "bb",  []),
         ("a",          "a",   [0])]
for s, p, exp in cases:
    r = sol438.findAnagrams(s, p)
    print(f"{'✅' if r==exp else '❌'}  s={s!r}, p={p!r} → {r} (期望 {exp})")

---
## Part 2 — 变长滑动窗口（求最长）

**框架**：扩张右指针，当窗口**不满足约束**时收缩左指针，每次扩张后更新答案。

```python
left = 0
for right in range(n):
    window 吸入 arr[right]            # 扩张
    while 约束不满足:                  # 违反约束 → 收缩
        window 吐出 arr[left]
        left += 1
    ans = max(ans, right - left + 1)  # 收缩后窗口一定合法，更新最长
```

**触发信号**：  
- "最长子数组/子串，满足某约束"
- 约束是"窗口内某指标 ≤ 限制"（如：无重复字符、最多 k 个 0 等）

### 3. LC 3 — 无重复字符的最长子串

| 项目 | 内容 |
|------|------|
| **题型** | 变长滑动窗口（求最长）+ 哈希表辅助 |
| **时间复杂度** | O(n) |
| **空间复杂度** | O(min(n, m))，m 为字符集大小 |
| **数据范围** | 0 ≤ s.length ≤ 5×10⁴，包含英文字母、数字、符号、空格 |

### ⚡ 触发条件
1. 最长**连续子串**满足某性质
2. 约束：窗口内**无重复字符**（每个字符最多出现一次）

### 🧠 算法（哈希表记录最后出现位置，O(n) 严格）
```
char_pos = {}   # char → 最后出现的索引
left = 0

for right in range(n):
    if s[right] in char_pos and char_pos[s[right]] >= left:
        left = char_pos[s[right]] + 1   # 跳过重复，直接定位
    char_pos[s[right]] = right
    max_len = max(max_len, right - left + 1)
```

**关键**：`char_pos[s[right]] >= left` 判断重复是否在窗口**内**（窗口外的旧记录不影响）。
直接令 `left = last_pos + 1` 比用 while 逐步移动更快（O(n) 而非 O(n²) 最坏）。

**对比**：set + while 写法（更直观但同样 O(n) 均摊）：
```python
while s[right] in char_set:
    char_set.remove(s[left]); left += 1
char_set.add(s[right])
```

In [ ]:
class Solution_3:
    def lengthOfLongestSubstring(self, s: str) -> int:
        char_pos = {}   # 字符 → 最近一次出现的索引
        left = 0
        max_len = 0

        for right in range(len(s)):
            # 若 s[right] 在窗口内出现过，直接跳过重复
            if s[right] in char_pos and char_pos[s[right]] >= left:
                left = char_pos[s[right]] + 1
            char_pos[s[right]] = right
            max_len = max(max_len, right - left + 1)

        return max_len

sol3 = Solution_3()
cases = [("abcabcbb", 3),   # "abc"
         ("bbbbb",    1),   # "b"
         ("pwwkew",   3),   # "wke"
         ("",         0),
         (" ",        1),
         ("dvdf",     3)]   # "vdf"
for s, exp in cases:
    r = sol3.lengthOfLongestSubstring(s)
    print(f"{'✅' if r==exp else '❌'}  {s!r} → {r} (期望 {exp})")

### 4. LC 1004 — 最大连续1的个数 III

| 项目 | 内容 |
|------|------|
| **题型** | 变长滑动窗口（求最长）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ nums.length ≤ 10⁵，nums[i] ∈ {0,1}，0 ≤ k ≤ nums.length |

### ⚡ 触发条件
1. 二进制数组，最多可翻转 **k 个 0 为 1**
2. 求翻转后**最长连续 1 的子数组**
3. 约束：窗口内 0 的个数 ≤ k

### 🧠 算法
```
left, zeros = 0, 0

for right in range(n):
    if nums[right] == 0: zeros += 1     # 扩张：记录 0 的数量
    while zeros > k:                     # 违反约束（0太多）→ 收缩
        if nums[left] == 0: zeros -= 1
        left += 1
    max_len = max(max_len, right - left + 1)
```

**理解**：将"翻转 k 个 0"转化为"窗口内至多 k 个 0"，典型的**约束转化**技巧。

**与 LC 3 对比**：
- LC 3 约束：窗口内无重复（用哈希表跟踪状态）
- LC 1004 约束：窗口内 0 的个数 ≤ k（用计数器跟踪状态）
- **框架完全相同**，只有"约束条件"和"状态变量"不同

In [ ]:
class Solution_1004:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left, zeros, max_len = 0, 0, 0

        for right in range(len(nums)):
            if nums[right] == 0:
                zeros += 1              # 扩张：纳入一个 0
            while zeros > k:            # 约束违反：0 太多
                if nums[left] == 0:
                    zeros -= 1
                left += 1
            max_len = max(max_len, right - left + 1)

        return max_len

sol1004 = Solution_1004()
cases = [([1,1,1,0,0,0,1,1,1,1,0],            2,  6),
         ([0,0,1,1,0,0,1,1,1,0,1,1,0,0,0,1,1,1,1], 3, 10),
         ([1,1,1,1],                           0,  4),   # k=0，全是1
         ([0,0,0],                             0,  0),   # k=0，无连续1
         ([0,0,0],                             3,  3)]   # k=3，全翻转
for nums, k, exp in cases:
    r = sol1004.longestOnes(nums, k)
    print(f"{'✅' if r==exp else '❌'}  k={k}, nums={nums} → {r} (期望 {exp})")

---
## Part 3 — 变长滑动窗口（求最短）

**框架**：扩张右指针，当窗口**满足条件**时记录答案并收缩左指针，尽量缩短。

```python
left = 0; ans = float('inf')
for right in range(n):
    window 吸入 arr[right]              # 扩张
    while 条件满足:                      # 已满足 → 尝试收缩
        ans = min(ans, right - left + 1) # 先记录（收缩前）
        window 吐出 arr[left]
        left += 1
return ans if ans != inf else 0
```

**触发信号**：  
- "满足某条件的**最短**连续子数组/子串"
- 条件是"窗口内某指标 ≥/= 目标"（如总和≥target、覆盖所有目标字符）

### 5. LC 209 — 长度最小的子数组

| 项目 | 内容 |
|------|------|
| **题型** | 变长滑动窗口（求最短）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ n ≤ 10⁵，**全为正整数**（保证单调性，是滑动窗口的前提）|

### ⚡ 触发条件
1. 连续子数组和 **≥ target**，求**最短长度**
2. 全正整数（收缩左边界时总和单调减小）

> ⚠️ 若有负数，收缩左边界不能保证总和减小 → **滑动窗口失效**，需改用前缀和。

In [ ]:
class Solution_209:
    def minSubArrayLen(self, target: int, nums: List[int]) -> int:
        lk, total, min_len = 0, 0, float('inf')
        for rk in range(len(nums)):
            total += nums[rk]
            while total >= target:              # 满足条件 → 先记录，再收缩
                min_len = min(min_len, rk - lk + 1)
                total -= nums[lk]; lk += 1
        return min_len if min_len != float('inf') else 0

sol209 = Solution_209()
cases = [(7,[2,3,1,2,4,3],2),(4,[1,4,4],1),(11,[1]*8,0),(3,[1]*5,3)]
for t, nums, exp in cases:
    r = sol209.minSubArrayLen(t, nums)
    print(f"{'✅' if r==exp else '❌'}  target={t}, {nums} → {r} (期望 {exp})")

### 6. LC 76 — 最小覆盖子串 ⭐（Hard）

| 项目 | 内容 |
|------|------|
| **题型** | 变长滑动窗口（求最短）+ 字符频率匹配 |
| **时间复杂度** | O(n + m)，n=len(s)，m=len(t) |
| **空间复杂度** | O(m)，字符集大小 |
| **数据范围** | 1 ≤ m, n ≤ 10⁵，大小写字母，t 可含重复字符 |

### ⚡ 触发条件
1. 在 s 中找包含 t **所有字符**（含重复）的**最短子串**
2. 需要高效判断"窗口是否覆盖 t"（用 `formed` 计数器，O(1) 判断）

### 🧠 算法（关键变量解释）

```
t_count  = Counter(t)        # t 中每个字符所需的数量
required = len(t_count)      # 需要满足的不同字符种数（去重后）
formed   = 0                  # 当前窗口中已满足计数的字符种数

for right in range(n):
    c = s[right]
    window[c] += 1
    # 当 c 的窗口计数恰好达到 t 中所需量时，formed +1
    if c in t_count and window[c] == t_count[c]:
        formed += 1

    while formed == required:    # 窗口已覆盖 t 的所有字符 → 尝试收缩
        更新最小窗口
        lc = s[left]
        window[lc] -= 1
        if lc in t_count and window[lc] < t_count[lc]:
            formed -= 1          # 去掉左字符后某个字符不再满足
        left += 1
```

**`formed` 的作用**：避免每次都比较整个 Counter（O(m)），改为 O(1) 判断窗口是否满足条件。  
`formed == required` 等价于"窗口包含 t 的所有字符（含重复计数）"。

**与 LC 209 的对比**：
- LC 209 条件：`total >= target`（数值比较，O(1)）
- LC 76 条件：`formed == required`（字符覆盖，O(1)，需维护 `formed`）
- **框架完全一致**，只是"满足条件"的判断方式不同

In [ ]:
class Solution_76:
    def minWindow(self, s: str, t: str) -> str:
        if not t or not s: return ""

        t_count  = Counter(t)
        required = len(t_count)   # 需要满足的不同字符种数

        window  = {}
        formed  = 0               # 当前已满足计数的字符种数
        left    = 0
        min_len, min_left = float('inf'), 0

        for right in range(len(s)):
            c = s[right]
            window[c] = window.get(c, 0) + 1
            # 该字符的窗口计数刚好达到 t 中所需量 → formed +1
            if c in t_count and window[c] == t_count[c]:
                formed += 1

            # 窗口已覆盖 t 的所有字符 → 先记录，再收缩
            while formed == required:
                if right - left + 1 < min_len:
                    min_len  = right - left + 1
                    min_left = left
                lc = s[left]
                window[lc] -= 1
                if lc in t_count and window[lc] < t_count[lc]:
                    formed -= 1   # 去掉左字符导致某字符不再满足
                left += 1

        return "" if min_len == float('inf') else s[min_left:min_left + min_len]

sol76 = Solution_76()
cases = [("ADOBECODEBANC", "ABC",  "BANC"),
         ("a",             "a",    "a"),
         ("a",             "aa",   ""),
         ("aa",            "aa",   "aa"),
         ("ADOBECODEBANC", "AABC", "ADOBEC")]  # t含重复A
for s, t, exp in cases:
    r = sol76.minWindow(s, t)
    ok = r == exp
    print(f"{'✅' if ok else '❌'}  s={s!r}, t={t!r} → {r!r} (期望 {exp!r})")

---
## 📋 总结：滑动窗口全模板 & 对照表

### 六题总览

| 题号 | 题型 | 时间 | 空间 | 约束/条件 | 状态变量 | 易错点 |
|------|------|------|------|---------|---------|-------|
| 643 | 固定窗口 | O(n) | O(1) | 窗口大小=k | window_sum | — |
| 438 | 固定窗口+频率 | O(n) | O(1) | 窗口大小=len(p) | Counter | 初始化首窗口别忘 |
| 3   | 变长-最长 | O(n) | O(m) | 无重复字符 | char_pos 字典 | char_pos[c]>=left 才算窗口内 |
| 1004 | 变长-最长 | O(n) | O(1) | 0的个数≤k | zeros 计数器 | 含负数时本方法失效 |
| 209 | 变长-最短 | O(n) | O(1) | 总和≥target，全正数 | total | while 非 if；含负数失效 |
| 76  | 变长-最短 | O(n+m) | O(m) | 覆盖 t 所有字符 | formed 计数器 | formed 的增减时机 |

---

### 🔁 三类模板汇总

```python
# ════ 固定窗口 ════════════════════════════════════════════
window_val = init(arr[:k])   # 初始化首个 k 元素窗口
for right in range(k, n):
    window_val += enter(arr[right])
    window_val -= leave(arr[right - k])
    更新答案

# ════ 变长窗口（求最长）════════════════════════════════════
left = 0
for right in range(n):
    window 吸入 arr[right]             # 扩张
    while 约束不满足:                   # 违反 → 收缩
        window 吐出 arr[left]; left += 1
    ans = max(ans, right - left + 1)   # 收缩后窗口合法，更新最长

# ════ 变长窗口（求最短）════════════════════════════════════
left = 0; ans = float('inf')
for right in range(n):
    window 吸入 arr[right]             # 扩张
    while 条件满足:                     # 已满足 → 先记录再收缩
        ans = min(ans, right - left + 1)
        window 吐出 arr[left]; left += 1
return ans if ans != float('inf') else 0
```

---

### 🗺️ 题型识别速查

```
"长度恰好 k 的子数组/子串的最值"          →  固定窗口         (LC 643)
"长度恰好 k 的固定窗口，字符频率匹配"     →  固定窗口+Counter  (LC 438)
"最长子串，约束：窗口内某属性 ≤ 上限"     →  变长窗口-求最长   (LC 3, LC 1004)
"最短子数组/子串，约束：窗口满足某目标"   →  变长窗口-求最短   (LC 209, LC 76)
⚠️ 数组含负数 + 要求子数组和             →  前缀和（不能用滑动窗口）
```

---
*复习笔记 · 2026-06 · By Claude*